Document question answering con HuggingFace
===========================================

En este notebook vamos a ejecutar un ejemplo mínimo de *document question answering*. La idea es tomar una imagen de un documento y hacerle una pregunta en lenguaje natural. A diferencia de un ejemplo de NLP tradicional, el modelo debe utilizar tanto la información textual como la información visual del documento.

## Preparación del ambiente

Instalemos las dependencias necesarias desde el archivo de requerimientos asociado a este notebook.

In [ ]:
!wget -q https://raw.githubusercontent.com/santiagxf/M72109/master/docs/document-understanding/document-question-answering.txt
%pip install -r document-question-answering.txt --quiet

Recordar reiniciar la sesión si se encuentra trabajando en Google Colab y el entorno lo solicita.

## Cargando un documento de ejemplo

Usaremos un pequeño conjunto de documentos de prueba disponible en HuggingFace. Cada registro contiene una imagen que podemos visualizar directamente.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("hf-internal-testing/example-documents", split="test")
documento = dataset[0]["image"]

documento

## Creando el pipeline

Para mantener el ejemplo compacto utilizaremos un pipeline de `transformers`. El modelo seleccionado está entrenado para responder preguntas sobre documentos. En este caso, el modelo procesa la imagen del documento y genera la respuesta como texto.

In [ ]:
from transformers import pipeline

qa_documentos = pipeline(
    "document-question-answering",
    model="naver-clova-ix/donut-base-finetuned-docvqa",
)

## Haciendo preguntas

Probemos con una pregunta simple. Note que la pregunta se escribe como texto, pero el contexto está en la imagen.

In [ ]:
pregunta = "What is the total?"

respuesta = qa_documentos(image=documento, question=pregunta)
respuesta

El resultado suele incluir el texto de la respuesta y, dependiendo del modelo, una puntuación de confianza. En un sistema real nos interesaría además conservar la página, el origen del documento y metadatos que permitan auditar de dónde salió la respuesta.

## Probando otra pregunta

Una buena práctica es evaluar el mismo documento con preguntas de distinto tipo: montos, fechas, nombres, direcciones o identificadores. Esto ayuda a detectar si el modelo está leyendo correctamente el documento o si solo responde cuando el formato es muy parecido a los ejemplos de entrenamiento.

In [ ]:
for pregunta in ["What is the date?", "What is the invoice number?"]:
    print("Pregunta:", pregunta)
    print("Respuesta:", qa_documentos(image=documento, question=pregunta))
    print("-" * 60)

## Cierre

Este ejemplo muestra el núcleo de la tarea: una imagen de documento entra al modelo junto con una pregunta y obtenemos una respuesta. Para llevarlo a producción habría que agregar validaciones, manejo de múltiples páginas, evaluación con documentos del dominio y posiblemente fine-tuning si los formatos son muy específicos.